## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [1]:
# The imports
from dotenv import load_dotenv
from agents import Agent, Runner, trace, ModelSettings
import os

In [2]:
# The usual starting point
load_dotenv(override=True)

True

In [3]:
# setup the model via LiteLLM model
! uv pip install git+https://github.com/BerriAI/litellm.git
from agents.extensions.models.litellm_model import LitellmModel

model = LitellmModel(
    model="openai/gpt-4.1-nano",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

Using Python 3.12.12 environment at: /home/martinsawojide/Desktop/Andela_AI_Eng/agents/.venv
   Updating https://github.com/BerriAI/litellm.git (HEAD)          
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  
   Updating https://github.com/BerriAI/litellm.git (HEAD)  

In [4]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor
from agents import set_tracing_disabled

# lauch phoenix for local tracing and visualization
session = px.launch_app()
set_tracing_disabled(True)                  # Disable native OpenAI tracing
print(f"🔍 Phoenix UI: {session.url}")

# Setup tracing
tracer_provider = register(project_name="andela_lab1", auto_instrument=True)
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

/home/martinsawojide/Desktop/Andela_AI_Eng/agents/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/martinsawojide/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/home/martinsawojide/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
Attempting to instrument while already instrumented


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔍 Phoenix UI: http://localhost:6006/
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: andela_lab1
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
# setup OpenAI Agent with LiteLLM model
agent = Agent(
    name="Joke Teller",
    instructions="An agent that tells jokes about Autonomous AI Agents",
    model=model
)

In [6]:
# Run the joke with Runner.run(agent, prompt) then print final_output

from opentelemetry import trace as otel_trace

# Get the tracer
tracer = otel_trace.get_tracer(__name__)

# Create a proper root span
with tracer.start_as_current_span("Telling a joke") as span:
    span.set_attribute("workflow.type", "joke_generation")
    span.set_attribute("task.name", "Telling a joke")
    
    result = await Runner.run(agent, "Tell me a joke")
    print(result.final_output)

Why did the autonomous AI agent go to therapy?

Because it had too many unresolved "issues" in its decision tree!


In [7]:
# from custom_tracing import traced_workflow

# @traced_workflow("Telling a joke", kind="CHAIN")
# async def tell_joke(agent, prompt):
#     result = await Runner.run(agent, prompt)
#     return result

# # Use it
# await tell_joke(agent, "Tell me a joke")

In [8]:
from custom_tracing import custom_trace

# Use as context manager
async with custom_trace("Telling a joke", kind="CHAIN", workflow_type="humor"):
    result = await Runner.run(agent, "Tell me a joke")
    print(result.final_output)

Why did the autonomous AI agent go to therapy?

Because it had too many unresolved loops!


## Now go and look at the trace

https://platform.openai.com/traces